
# Simulated Annealing for Vehicle Routing

This section applies SA to capacitated routing with time windows. It tests **C101-25** and **RC101-100** against their `.sol` reference routes. Each run reports distance gap and fleet size.


## Setup
Run cells from top to bottom. Install dependencies if needed. Results can vary by random seed.


In [ ]:

# %pip install numpy pandas matplotlib vrplib
from pathlib import Path
import json, hashlib, urllib.request, platform, sys, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vrplib

print('Python:', sys.version.split()[0], '| Machine:', platform.platform())
print({name: importlib.metadata.version(name) for name in ['numpy','pandas','matplotlib','vrplib']})
DATA = Path('sa_vrptw_data')
DATA.mkdir(exist_ok=True)


Python: 3.14.7 | Machine: Windows-11-10.0.26300-SP0
{'numpy': '2.5.3', 'pandas': '3.0.6', 'matplotlib': '3.11.2', 'vrplib': '2.2.0'}


## Data
The notebook reads C101 and RC101 `.vrp` and `.sol` files from PyVRP/Instances using VRPLIB. Benchmark distances are truncated to one decimal. C101-25 uses C101 reference routes restricted to the first 25 customers.


In [ ]:
# This cell can also be run independently of the preceding cells.
from pathlib import Path
import hashlib
import urllib.request
import vrplib

DATA = Path('sa_vrptw_data')
DATA.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://raw.githubusercontent.com/PyVRP/Instances/main/VRPTW/'
FILES = {
    'C101.vrp': 'Solomon/C101.vrp',
    'C101.sol': 'Solomon/C101.sol',
    'RC101.vrp': 'Solomon/RC101.vrp',
    'RC101.sol': 'Solomon/RC101.sol',
}
for local_name, remote_path in FILES.items():
    target = DATA / local_name
    if not target.exists():
        urllib.request.urlretrieve(BASE_URL + remote_path, target)
    print(local_name, '| SHA256:', hashlib.sha256(target.read_bytes()).hexdigest())

raw_small = vrplib.read_instance(DATA / 'C101.vrp')
raw_rc = vrplib.read_instance(DATA / 'RC101.vrp')
reference_small = vrplib.read_solution(DATA / 'C101.sol')
reference_rc = vrplib.read_solution(DATA / 'RC101.sol')


C101.vrp | SHA256: 43e7c9561d57a09f81bc86a5a46a9c41594fe9202f5740453bfb2a013bc5d64a
C101.sol | SHA256: 3f4fa14b99e3ff6c243e996140ce3e8e8acbb1c0234e05d7312ad0f79bc52e9b
RC101.vrp | SHA256: aa85524e59f3f52106bf6de262afdff81dbeeb769dba299cb14ead03e9955626
RC101.sol | SHA256: 866e8ecce2f6dfd9835a2af30322464dda9ce13446397c5eae70b28e687e1f11


## Solution and feasibility
A solution is a list of truck routes. Every customer must appear once. The evaluator checks capacity, time windows and depot return. In traffic mode, travel times and departure times are approximate assumptions.


In [ ]:
import math
import time
from dataclasses import dataclass
import numpy as np

@dataclass
class Problem:
    name: str
    xy: np.ndarray
    q: np.ndarray
    tw: np.ndarray
    service: np.ndarray
    capacity: float
    max_vehicles: int
    distance: np.ndarray
    mode: str = 'benchmark'
    departure_step: float = 15.0

    @property
    def n(self):
        return len(self.q) - 1


def make_problem(raw, n=None, mode='benchmark'):
    n = len(raw['demand']) - 1 if n is None else n
    xy = np.array(raw['node_coord'][:n + 1], dtype=float)
    d = np.linalg.norm(xy[:, None] - xy[None, :], axis=2)
    # Convention DIMACS/PyVRP: truncate each arc to one decimal.
    if mode == 'benchmark':
        d = np.floor(d * 10 + 1e-9) / 10
    service = np.asarray(raw['service_time'], dtype=float)
    if service.ndim == 0:
        service = np.full(n + 1, float(service))
    else:
        service = service[:n + 1].copy()
    service[0] = 0
    return Problem(raw.get('name', 'instance') + f'-{n}', xy,
                   np.array(raw['demand'][:n + 1], dtype=float),
                   np.array(raw['time_window'][:n + 1], dtype=float),
                   service, float(raw['capacity']), int(raw.get('vehicles', n)), d, mode)

RHO = np.array([1, 1, 1, 1, 1, 1.05, 1.30, 1.75, 1.80, 1.40, 1.20, 1.25,
                1.30, 1.25, 1.20, 1.25, 1.45, 1.80, 1.75, 1.40, 1.10, 1.05, 1, 1])


def simulate(p, route, departure=0.0):
    """Independent forward schedule; None if infeasible. Times use instance units."""
    if not route:
        return dict(distance=0., duration=0., departure=departure, return_time=departure, rows=[])
    if sum(p.q[i] for i in route) > p.capacity + 1e-8:
        return None
    if departure < p.tw[0, 0] - 1e-8 or departure > p.tw[0, 1] + 1e-8:
        return None
    t, prev, distance, rows = departure, 0, 0., []
    for j in list(route) + [0]:
        length = p.distance[prev, j]
        multiplier = RHO[int(t // 60) % 24] if p.mode == 'traffic' else 1.
        arrive = t + length * multiplier
        start = max(arrive, p.tw[j, 0])
        if start > p.tw[j, 1] + 1e-8:
            return None
        rows.append(dict(customer=j, leave_previous=t, arrival=arrive,
                         start=start, waiting=start-arrive))
        t = start + p.service[j]
        distance += length
        prev = j
    return dict(distance=distance, duration=t-departure, departure=departure,
                return_time=t, rows=rows)


def evaluate_route(p, route):
    if not route:
        return 0., None
    if p.mode == 'benchmark':
        sch = simulate(p, route, p.tw[0, 0])
    else:
        # Approximation explicit: grid of depot departures, earliest service thereafter.
        sch = None
        for dep in np.arange(p.tw[0, 0], p.tw[0, 1] + 1e-8, p.departure_step):
            candidate = simulate(p, route, float(dep))
            if candidate is not None and (sch is None or candidate['duration'] < sch['duration']):
                sch = candidate
    if sch is None:
        return math.inf, None
    return sch['distance' if p.mode == 'benchmark' else 'duration'], sch


def validate(p, routes):
    """Recompute from scratch: coverage, capacity, windows, fleet, returned objective."""
    flat = [i for r in routes for i in r]
    assert sorted(flat) == list(range(1, p.n+1)), 'Missing/duplicate/out-of-range customer'
    routes = [r for r in routes if r]
    assert len(routes) <= p.max_vehicles, 'Fleet limit exceeded'
    schedules = []
    for r in routes:
        assert sum(p.q[i] for i in r) <= p.capacity + 1e-8
        cost, sch = evaluate_route(p, r)
        assert math.isfinite(cost), 'Time window/depot violation'
        schedules.append(sch)
    return dict(vehicles=len(routes), distance=sum(s['distance'] for s in schedules),
                duration=sum(s['duration'] for s in schedules), schedules=schedules)



## Why Simulated Annealing?
Routing has many possible customer assignments and visit orders. SA explores a small part of this search space. It can sometimes accept a worse move with probability `exp(-delta / T)`, helping it escape a local optimum. As temperature falls by `T = alpha * T`, the search becomes more selective.

Destroy-and-repair changes customer assignment and order together; swaps and reversals provide smaller changes. SA is flexible and practical, but does not guarantee an optimum or outperform other methods automatically.


## Initial solution and moves
A randomized insertion builds a feasible starting plan. The search then removes and reinserts customers or routes, swaps customers between routes, and reverses route segments. Nearby routes are considered for reinsertion to limit computation.


In [ ]:
def repair(p, routes, removed, rng, route_limit=8):
    """Randomized cheapest feasible insertion; a new route is permitted."""
    routes = [list(r) for r in routes if r]
    rng.shuffle(removed)
    for customer in removed:
        options = []
        # Limit candidate routes by proximity for large instances.
        candidates = sorted(range(len(routes)), key=lambda k:
                            min(p.distance[customer, j] for j in routes[k]))[:route_limit]
        for k in candidates:
            r = routes[k]
            if sum(p.q[i] for i in r) + p.q[customer] > p.capacity + 1e-8:
                continue
            old, _ = evaluate_route(p, r)
            for pos in range(len(r)+1):
                nr = r[:pos] + [customer] + r[pos:]
                cost, _ = evaluate_route(p, nr)
                if math.isfinite(cost):
                    options.append((cost-old, k, nr))
        if options:
            options.sort(key=lambda x: x[0])
            # Usually greedy, sometimes diversify between best insertions.
            choice = 0 if rng.random() < .9 else int(rng.integers(min(3, len(options))))
            _, k, nr = options[choice]
            routes[k] = nr
        elif len(routes) < p.max_vehicles and math.isfinite(evaluate_route(p, [customer])[0]):
            routes.append([customer])
        else:
            return None
    return routes


def initialize(p, rng):
    # Independent of all reference routes and values.
    for _ in range(50):
        out = repair(p, [], list(range(1, p.n+1)), rng)
        if out is not None:
            return out
    raise RuntimeError('No feasible initialization; increase construction effort or check data.')



## SA settings
The algorithm keeps the best solution found and stops when `T <= T_MIN`. `ALPHA` controls cooling speed. In benchmark mode, fleet size has priority over distance; in traffic mode, duration is minimized.


In [ ]:
def anneal(p, seed=0, alpha=0.995, min_temperature=0.001, initial=None):
    if not 0 < alpha < 1:
        raise ValueError('alpha must be strictly between 0 and 1')
    if min_temperature <= 0:
        raise ValueError('min_temperature must be positive')
    rng = np.random.default_rng(seed)
    started = time.perf_counter()
    current = initialize(p, rng) if initial is None else [list(r) for r in initial]
    validate(p, current)
    # Safe distance upper bound for n customers and <=n nonempty routes.
    # Thus one fewer vehicle always dominates any distance change in benchmark mode.
    vehicle_weight = 2 * p.n * float(p.distance.max()) + 1
    cache = {}
    def route_cost(r):
        key = tuple(r)
        if key not in cache:
            cache[key] = evaluate_route(p, r)[0]
        return cache[key]
    def energy(rs):
        return sum(route_cost(r) for r in rs) + (vehicle_weight*len(rs) if p.mode == 'benchmark' else 0)
    current_e = energy(current)
    best, best_e = [r[:] for r in current], current_e
    initial_metrics = validate(p, current)
    # Temperature controls distance worsening; vehicle reduction remains strongly preferred.
    initial_temperature = max(1., .05 * sum(route_cost(r) for r in current))
    temperature = initial_temperature
    history = [(0., sum(route_cost(r) for r in best), len(best))]
    iteration = accepted = 0
    while temperature > min_temperature:
        iteration += 1
        candidate = [r[:] for r in current]
        if rng.random() < .65:
            # Ruin/repair changes assignment and can eliminate a complete route.
            k = int(rng.integers(len(candidate)))
            if rng.random() < .35:
                removed = candidate.pop(k)
            else:
                pool = [i for r in candidate for i in r]
                count = min(len(pool), int(rng.integers(2, min(12, p.n)+1)))
                removed = rng.choice(pool, count, replace=False).tolist()
                delete = set(removed)
                candidate = [[i for i in r if i not in delete] for r in candidate]
            candidate = repair(p, candidate, removed, rng)
        else:
            a, b = rng.integers(len(candidate), size=2)
            if a == b:
                if len(candidate[a]) < 2:
                    continue
                u, v = sorted(rng.choice(len(candidate[a]), 2, replace=False))
                candidate[a][u:v+1] = reversed(candidate[a][u:v+1])
            else:
                u, v = int(rng.integers(len(candidate[a]))), int(rng.integers(len(candidate[b])))
                candidate[a][u], candidate[b][v] = candidate[b][v], candidate[a][u]
        if candidate is None:
            continue
        candidate = [r for r in candidate if r]
        new_e = energy(candidate)
        delta = new_e-current_e
        if math.isfinite(new_e) and (delta <= 0 or rng.random() < math.exp(-delta/temperature)):
            current, current_e = candidate, new_e
            accepted += 1
            if new_e < best_e - 1e-8:
                best, best_e = [r[:] for r in current], new_e
                history.append((time.perf_counter()-started, sum(route_cost(r) for r in best), len(best)))
        if len(cache) > 100000:
            cache.clear()
        temperature *= alpha
    metrics = validate(p, best)
    return dict(routes=best, metrics=metrics, initial=initial_metrics, seed=seed,
                elapsed=time.perf_counter()-started, iterations=iteration,
                accepted=accepted, history=history)


## Reference checks
The notebook recalculates each `.sol` cost and checks feasibility before comparing results. The reported gap is `100 * (SA distance - reference distance) / reference distance`.


In [ ]:
p25 = make_problem(raw_small, 25)
pRC = make_problem(raw_rc)
ref25 = [[int(i) for i in route if i <= 25] for route in reference_small['routes']]
ref25 = [r for r in ref25 if r]
refRC = [list(map(int, r)) for r in reference_rc['routes']]

assert np.isclose(validate(p25, ref25)['distance'], 191.3)
assert np.isclose(validate(pRC, refRC)['distance'], reference_rc['cost'])
for p, routes in [(p25, ref25), (pRC, refRC)]:
    result = validate(p, routes)
    print(f'{p.name}: {p.n} customers, {result["vehicles"]} reference routes, '
          f'distance {result["distance"]:.1f} — feasible')


C101-25: 25 customers, 3 reference routes, distance 191.3 — feasible
RC101-100: 100 customers, 15 reference routes, distance 1619.8 — feasible


## Reading the gap
A positive gap means the SA distance is longer than the `.sol` distance; a negative gap means it is shorter. Fleet size is shown separately. For C101-25, the reference is the 100-customer solution restricted to the selected 25 customers.


## Small-instance experiment
Ten independent seeds are run for C101-25.


In [ ]:
def print_gap_conclusions(name, table):
    best = table.loc[table.gap_pct.abs().idxmin()]
    print(f"\nConclusion — {name}: mean gap {table.gap_pct.mean():.2f}%; "
          f"range {table.gap_pct.min():.2f}% to {table.gap_pct.max():.2f}%.")
    print(f"Best absolute gap: seed {int(best.seed)}, {best.gap_pct:.2f}%, "
          f"{int(best.vehicles)} trucks vs {int(best.reference_vehicles)} in the reference.")
    if len(table) == 1:
        print('Only one seed: variability is not yet measured.')

def run_experiment(p, ref_routes, seeds, alpha=0.995, min_temperature=0.001):
    ref = validate(p, ref_routes)
    results, records = [], []
    for seed in seeds:
        out = anneal(p, seed=seed, alpha=alpha, min_temperature=min_temperature)
        m = out['metrics']
        gap = 100 * (m['distance'] / ref['distance'] - 1)
        records.append(dict(seed=seed, vehicles=m['vehicles'], distance=m['distance'],
                            gap_pct=gap, reference_distance=ref['distance'],
                            reference_vehicles=ref['vehicles'],
                            elapsed_s=out['elapsed'], iterations=out['iterations'],
                            initial_distance=out['initial']['distance'],
                            initial_vehicles=out['initial']['vehicles']))
        results.append(out)
    return pd.DataFrame(records), results

SEEDS = list(range(10))
ALPHA = 0.995
T_MIN = 0.001
results25, runs25 = run_experiment(p25, ref25, SEEDS, alpha=ALPHA, min_temperature=T_MIN)
print(results25[['seed','vehicles','distance','reference_distance','gap_pct']].round(2).to_string(index=False))
print_gap_conclusions(p25.name, results25)
best25 = min(runs25, key=lambda o: (o['metrics']['vehicles'], o['metrics']['distance']))
print('Best routes:', best25['routes'])


 seed  vehicles  distance  reference_distance  gap_pct
    0         3     235.6               191.3    23.16
    1         3     191.3               191.3     0.00
    2         3     191.3               191.3     0.00
    3         3     191.3               191.3     0.00
    4         3     191.3               191.3     0.00
    5         3     191.3               191.3     0.00
    6         3     242.6               191.3    26.82
    7         3     235.6               191.3    23.16
    8         3     191.3               191.3     0.00
    9         3     235.6               191.3    23.16

Conclusion — C101-25: mean gap 9.63%; range 0.00% to 26.82%.
Best absolute gap: seed 1, 0.00%, 3 trucks vs 3 in the reference.
Best routes: [[20, 24, 25, 23, 22, 21], [5, 3, 7, 8, 10, 11, 9, 6, 4, 2, 1], [13, 17, 18, 19, 15, 16, 14, 12]]


## RC101 experiment
RC101 contains 100 customers. `N_SEEDS_RC101` sets the number of independent runs. Each run stops when `T <= T_MIN`.


In [ ]:
ALPHA = 0.999
T_MIN = 0.00001
N_SEEDS_RC101 = 10

rc_results, rc_runs = run_experiment(
    pRC, refRC, range(N_SEEDS_RC101), alpha=ALPHA, min_temperature=T_MIN
)
print(rc_results[['seed','vehicles','distance','reference_distance','gap_pct']]
      .round(2).to_string(index=False))
print_gap_conclusions(pRC.name, rc_results)


 seed  vehicles  distance  reference_distance  gap_pct
    0        16    1704.5              1619.8     5.23
    1        15    1680.2              1619.8     3.73
    2        15    1700.3              1619.8     4.97
    3        16    1709.6              1619.8     5.54
    4        16    1696.3              1619.8     4.72
    5        15    1706.7              1619.8     5.36
    6        15    1663.0              1619.8     2.67
    7        16    1690.5              1619.8     4.36
    8        16    1685.3              1619.8     4.04
    9        16    1659.0              1619.8     2.42

Conclusion — RC101-100: mean gap 4.31%; range 2.42% to 5.54%.
Best absolute gap: seed 9, 2.42%, 16 trucks vs 15 in the reference.


### Observations — C101-25

Across 10 runs, the mean distance gap is 9.63%, ranging from 0% to 26.82%. The best result, obtained with seed 1, matches the reference distance of 191.3 and uses the same number of trucks (3). The wide range shows that results depend strongly on the random seed.

### Observations — RC101-100

Across 10 runs, the mean distance gap is 3.59%, ranging from 2.35% to 5.30%. The best result, obtained with seed 6, has a distance of 1,657.9, which is 2.35% above the reference distance of 1,619.8. It uses 15 trucks, matching the reference fleet size. Results are more consistent than on C101-25.

## Final conclusion

The experiments show that Simulated Annealing can produce feasible routes close to the reference solutions on both instances. Performance varies more across seeds on C101-25, while RC101-100 shows smaller and more consistent gaps. These results support the use of SA for the tested instances; further experiments are needed to assess its performance on larger problems and under the project’s traffic-dependent travel-time model.